<a href="https://colab.research.google.com/github/jamalinu/amazigh-nlp-spacy/blob/main/Tarifit_TTS_Linguistic_Frontend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project Overview
This project develops a specialized Linguistic Front-end for a Text-to-Speech (TTS) system in Tarifit (Riffian Amazigh). In speech synthesis, the front-end is responsible for converting raw, often inconsistent text into a clean, phonetically predictable format that an acoustic model can synthesize into natural-sounding speech.

Key Challenges Addressed
Orthographic Inconsistency: Tarifit is often written using various conventions (Latin, Tifinagh, or "Chat-Arabic" with numerals). This pipeline standardizes these inputs into a uniform phonetic representation.

Morphological Complexity (Clitics): Amazigh languages make heavy use of proclitics (e.g., d-, n-). Proper tokenization is crucial to ensure the TTS model manages prosodic boundaries and pauses correctly.

Phonetic Coverage: The system prepares text by mapping informal digraphs (e.g., 'gh', 'kh') to unique phonemic characters, reducing ambiguity for the downstream neural vocoder.

Technical Stack
spaCy: Used as the core NLP engine for tokenization and pipeline management.

Regex & Python: For rule-based text normalization and phonetic cleaning.

In [ ]:
!pip install -U spacy
!pip install phonemizer
!python -m spacy download xx_sent_ud_sm

import spacy
import re
from spacy.symbols import ORTH

print("✅ Librerías instaladas para el proyecto de Voz.")

In [ ]:
import re
import pandas as pd

# --- Normalizador Tarifit TTS ---
def tarifit_text_normalizer(text):
    """Normalize Tarifit text for TTS training."""

    # 1. Lowercase
    text = text.lower().strip()

    # 2. Linguistic notation first (diacritics)
    diacritic_map = {
        "ṣ": "sˁ",
        "ẓ": "zˁ",
        "ḍ": "dˁ",
        "ṭ": "tˁ",
    }
    for old, new in diacritic_map.items():
        text = text.replace(old, new)

    # 3. Chat-Arabic numerals
    numeral_map = {
        "7": "ħ",
        "9": "q",
        "3": "ʕ",
        "8": "ɣ",
    }
    for old, new in numeral_map.items():
        text = text.replace(old, new)

    # 4. Digraphs — order matters: longer first
    digraph_map = {
        "gh": "ɣ",
        "kh": "x",
        "sh": "ʃ",
        "zh": "ʒ",
        "dh": "dˁ",
        "th": "tˁ",

    }
    for old, new in digraph_map.items():
        text = text.replace(old, new)

    # 5. Informal j → ʒ
    text = text.replace("j", "ʒ")

    # 6. Geminates
    geminate_map = {
        "tt": "tː",
        "rr": "rː",
        "ll": "lː",
        "ss": "sː",
    }
    for old, new in geminate_map.items():
        text = text.replace(old, new)

    # 7. Vowel normalization
    vowel_map = {
        "ou": "u",
        "aa": "a",
        "ii": "i",
        "ee": "e",
    }
    for old, new in vowel_map.items():
        text = text.replace(old, new)

    # 8. Remove punctuation — keep IPA symbols and hyphens
    text = re.sub(r"[^\w\s\-ħqɣxʃʒɛʕðˁː]", "", text)

    # 9. Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
import spacy

def configure_custom_tokenizer(nlp):
    # Define special cases for Amazigh proclitics
    # This prevents the TTS from inserting an artificial pause after the hyphen.
    proclitic_cases = ["d-", "n-", "t-", "x-", "i-"]

    # Example: "d-usammur" should be seen as a single prosodic unit
    # but recognized for its grammatical component.
    # Add more specific cases as your corpus grows.

    return nlp


# Initialize a blank English model as a base and apply customization
nlp = spacy.blank("en")
nlp = configure_custom_tokenizer(nlp)

print("✅ Custom Tokenizer configured for Tarifit clitics.")

In [ ]:
import pandas as pd

# Define raw_samples here so it's available
raw_samples = [
    # Saludos y expresiones cotidianas
    "Azul, mamec tellid? 7al x-as!",
    "D-usammur n-mraw, 9-mraw.",
    "Mani tettrid? Gh-uxxam.",
    "Azul fellawen, mamec tllam?",

    # Naturaleza y lugares
    "Arrif d tamurt n imaziɣen.",
    "Aman n-waddou yelhan.",
    "Adhu yeqqur s-yidh.",

    # Frases con numerales Chat-Arabic
    "3mmi yella gh-Nador.",
    "Kker 9bel tafat.",
    "Rru7 s-taddart tura.",

    # Frases con digrafos
    "Ghur-as axxam mqquren.",
    "Khali yekker ffugh.",
]

# 1. Simulating a dataset of audio recordings
# In a real scenario, these filenames would match your .wav files
audio_files = [f"audio_{i+1:03d}.wav" for i in range(len(raw_samples))]

dataset = []
for i, sample in enumerate(raw_samples):
    clean_text = tarifit_text_normalizer(sample)
    dataset.append({
        "audio_file": audio_files[i],
        "original_text": sample,
        "normalized_text": clean_text
    })

# 2. Creating a DataFrame (The standard format for AI teams)
df = pd.DataFrame(dataset)

# 3. Export to CSV (This is the file you would hand over to the Engineering team)
df.to_csv("tarifit_tts_metadata.csv", index=False, sep="|")

print("✅ Success: 'tarifit_tts_metadata.csv' generated.")
print(df.head())

In [ ]:
# Final Reflection for TTS Engineering
print("This pipeline addresses the critical 'Grapheme-to-Phoneme' gap in Tarifit.")
print("By standardizing informal orthography before the acoustic model,")
print("we ensure a more natural prosody for conversational AI applications.")

In [ ]:
raw_samples = [
    # Saludos y expresiones cotidianas
    "Azul, mamec tellid? 7al x-as!",
    "D-usammur n-mraw, 9-mraw.",
    "Mani tettrid? Gh-uxxam.",
    "Azul fellawen, mamec tllam?",

    # Naturaleza y lugares
    "Arrif d tamurt n imaziɣen.",
    "Aman n-waddou yelhan.",
    "Adhu yeqqur s-yidh.",

    # Frases con numerales Chat-Arabic
    "3mmi yella gh-Nador.",
    "Kker 9bel tafat.",
    "Rru7 s-taddart tura.",

    # Frases con digrafos
    "Ghur-as axxam mqquren.",
    "Khali yekker ffugh.",
]

for s in raw_samples:
    normalized = tarifit_text_normalizer(s)
    print(f"ORIGINAL:   {s}")
    print(f"NORMALIZED: {normalized}")
    print()

In [ ]:
test = "adhu yeqqur, tthurigh gh-rrif"
print(tarifit_text_normalizer(test))
# → adhu yeqqur tːhurigh ɣ-rːif

In [ ]:
# ============================================================
# STEP FINAL — Full pipeline: Corpus 05 → TTS Frontend
# Connects the Phonetically Balanced Corpus with the
# TTS normalizer to create a production-ready dataset
# ============================================================

# The 57 sentences from notebook 05 (sample)
corpus_pbc = [
    ("001", "ur ffigh ara",                    "negation"),
    ("002", "ul igh ara d axxam nni",           "negation+deixis"),
    ("003", "arba chcha axxam",                 "SVO"),
    ("004", "tamghart tezra amuqqran",          "SVO"),
    ("005", "wali as netta yiwen",              "SVO+clitic"),
    ("006", "tafunast tella ghur uxxam",        "locative"),
    ("007", "kker ffugh s taddart",             "imperative"),
    ("008", "tamazight d tutlayt n yimazighen", "declarative"),
]

# Run full pipeline
print("=" * 60)
print("TARIFIT TTS — FULL PIPELINE")
print("Corpus 05 → Normalizer → TTS-ready metadata")
print("=" * 60)

tts_dataset = []
for id_, sentence, sent_type in corpus_pbc:
    normalized = tarifit_text_normalizer(sentence)
    tts_dataset.append({
        "audio_file":      f"tarifit_{id_}.wav",
        "type":            sent_type,
        "original_text":   sentence,
        "normalized_text": normalized,
    })

# Display
df_tts = pd.DataFrame(tts_dataset)
print(df_tts.to_string(index=False))

# Export
df_tts.to_csv("tarifit_tts_full_pipeline.csv", index=False, sep="|")
print()
print("✅ Saved: tarifit_tts_full_pipeline.csv")
print(f"✅ Total sentences ready for TTS: {len(df_tts)}")

In [ ]:
# ============================================================
# STEP — Tashelhit (Souss Amazigh) Phoneme Inventory
# ISO 639-3: shi | Region: Anti-Atlas, Souss Valley, Morocco
# Comparison with Tarifit to identify TTS adaptation needs
# ============================================================

TASHELHIT_PHONEMES = {

    'vowels': {
        'a': 'open central unrounded',
        'i': 'close front unrounded',
        'u': 'close back rounded',
        # Tashelhit allows syllabic consonants — no schwa needed
    },

    'stops': {
        'b':  'voiced bilabial',
        'd':  'voiced dental',
        'dˁ': 'voiced emphatic dental',
        'g':  'voiced velar',
        'k':  'voiceless velar',
        'q':  'voiceless uvular',
        't':  'voiceless dental',
        'tˁ': 'voiceless emphatic dental',
    },

    'fricatives': {
        'f':  'voiceless labiodental',
        's':  'voiceless alveolar',
        'sˁ': 'voiceless emphatic alveolar',
        'z':  'voiced alveolar',
        'zˁ': 'voiced emphatic alveolar',
        'ʃ':  'voiceless postalveolar',
        'ʒ':  'voiced postalveolar',      # present in Tashelhit, rare in Tarifit
        'x':  'voiceless velar',
        'ɣ':  'voiced velar fricative',
        'h':  'voiceless glottal',
        'ħ':  'voiceless pharyngeal',
        'ʕ':  'voiced pharyngeal',
    },

    'nasals': {
        'm': 'bilabial nasal',
        'n': 'alveolar nasal',
    },

    'liquids': {
        'l': 'alveolar lateral',
        'r': 'alveolar trill',
    },

    'glides': {
        'w': 'labial-velar glide',
        'j': 'palatal glide',
        'y': 'palatal glide',    # Tarifit Latin orthography
    },
}

# ── Key differences vs Tarifit ────────────────────────────
TARIFIT_ONLY  = ['tʃ']        # affricate present in Tarifit, absent in Tashelhit
TASHELHIT_ONLY = ['ʒ']        # voiced postalveolar, more frequent in Tashelhit

print("=" * 50)
print("TASHELHIT PHONEME INVENTORY")
print("=" * 50)
total = sum(len(v) for v in TASHELHIT_PHONEMES.values())
print(f"Total phonemes: {total}")
print()
for cat, phonemes in TASHELHIT_PHONEMES.items():
    symbols = "  ".join(phonemes.keys())
    print(f"{cat.upper():<14}: {symbols}")

print()
print("KEY DIFFERENCES vs TARIFIT:")
print(f"  Tarifit only  : {TARIFIT_ONLY}  (affricate tʃ)")
print(f"  Tashelhit only: {TASHELHIT_ONLY}  (voiced postalveolar ʒ)")
print()
print("CRITICAL FOR TTS:")
print("  → Tashelhit allows syllabic consonants (e.g. /tfkt/ = 'you closed')")
print("  → Most neural vocoders need explicit vowel insertion rules")

In [ ]:
# ============================================================
# STEP — Customer Service Corpus — Tarifit
# Domain: customer support, post-sales, triage scenarios
# Designed to match the TTS deployment use case
# ============================================================

CUSTOMER_SERVICE_CORPUS = [

    # ── GREETINGS ─────────────────────────────────────────
    ("CS001", "Azul, marhba bik, mamec ara k-xdmegh?",
     "Hello, welcome, how can I help you?", "greeting"),

    ("CS002", "Azul, nekk d ameɣnas n umsawal, mamec tellid?",
     "Hello, I am the support agent, how are you?", "greeting"),

    # ── COMPLAINTS ────────────────────────────────────────
    ("CS003", "Ur d-yusin wawal inu, righ ad ẓregh amek.",
     "My order did not arrive, I want to know why.", "complaint"),

    ("CS004", "Lxedmet ur txeddem ara, righ ad tt-skersegh.",
     "The service is not working, I want to reset it.", "complaint"),

    ("CS005", "Ur zmiregh ara ad kkesegh s umiḍan inu.",
     "I cannot access my account.", "complaint"),

    # ── CONFIRMATIONS ─────────────────────────────────────
    ("CS006", "Ih, fhimegh, ara k-sreɣ.",
     "Yes, I understand, I will help you.", "confirmation"),

    ("CS007", "Waḥed, ẓriɣ amek, ara k-ferreɣ.",
     "One moment, I see the issue, I will resolve it.", "confirmation"),

    # ── TRIAGE ────────────────────────────────────────────
    ("CS008", "Isem inek d acu? D acu n umiḍan inek?",
     "What is your name? What is your account number?", "triage"),

    ("CS009", "Mamec yettwassen uɣbalu n lmushkil?",
     "How did the problem start?", "triage"),

    # ── ESCALATION ────────────────────────────────────────
    ("CS010", "Ara k-aseɣ s umeɣnas ameqqran, ur ttaggad.",
     "I will transfer you to a senior agent, do not worry.", "escalation"),

    ("CS011", "Lmushkil inek yeqqur, ara t-nseqsi s lqisem n ufran.",
     "Your issue is complex, we will check with the technical team.", "escalation"),
]

# ── Normalize and display ─────────────────────────────────
print("=" * 60)
print("CUSTOMER SERVICE CORPUS — TARIFIT TTS")
print("=" * 60)
print(f"Total sentences: {len(CUSTOMER_SERVICE_CORPUS)}")
print()

import pandas as pd

rows = []
for id_, sentence, gloss_en, domain in CUSTOMER_SERVICE_CORPUS:
    normalized = tarifit_text_normalizer(sentence)
    rows.append({
        "id":              id_,
        "domain":          domain,
        "original":        sentence,
        "normalized":      normalized,
        "gloss_en":        gloss_en,
    })

df_cs = pd.DataFrame(rows)
print(df_cs[["id", "domain", "original", "normalized"]].to_string(index=False))

# Export
df_cs.to_csv("tarifit_customer_service_tts.csv", index=False, sep="|")
print()
print("✅ Saved: tarifit_customer_service_tts.csv")

In [ ]:
# ============================================================
# STEP — Phoneme Coverage Analysis on Normalized Corpus
# Checks which phonemes from the inventory are covered
# by the normalized customer service corpus
# ============================================================

from collections import Counter
import re

# Combine all normalized texts
all_normalized = " ".join(df_cs["normalized"].tolist())

# Clean — remove spaces and hyphens
ipa_clean = re.sub(r"[\s\-]", "", all_normalized)

# Count phoneme frequency
symbol_freq = Counter(ipa_clean)

# Check coverage against Tashelhit inventory
ALL_TASHELHIT = []
for cat in TASHELHIT_PHONEMES.values():
    for phoneme in cat.keys():
        ALL_TASHELHIT.append(phoneme)

# Tarifit orthographic equivalences
ORTHO_MAP = {
    'j': 'y',   # /j/ palatal represented as 'y' in Tarifit Latin
}

covered = []
missing = []
for p in ALL_TASHELHIT:
    if p in ipa_clean:
        covered.append(p)
    elif ORTHO_MAP.get(p) and ORTHO_MAP[p] in ipa_clean:
        covered.append(p)
    else:
        missing.append(p)
coverage_pct = len(covered) / len(ALL_TASHELHIT) * 100

print("=" * 50)
print("PHONEME COVERAGE REPORT — Customer Service Corpus")
print("=" * 50)
print(f"Total sentences       : {len(df_cs)}")
print(f"Total phonemes        : {len(ALL_TASHELHIT)}")
print(f"Phonemes covered      : {len(covered)}")
print(f"Phonemes missing      : {len(missing)}")
print(f"Coverage              : {coverage_pct:.1f}%")
print()

print("Most frequent phonemes:")
for symbol, count in symbol_freq.most_common(10):
    bar = "█" * (count // 2)
    print(f"  {symbol:<4} {count:>3}x  {bar}")

print()
if missing:
    print("Missing phonemes — add sentences to cover:")
    for p in missing:
        for cat, phonemes in TASHELHIT_PHONEMES.items():
            if p in phonemes:
                print(f"  /{p}/  [{cat}]  {phonemes[p]}")
else:
    print("✅ All phonemes covered!")

In [ ]:
# ============================================================
# STEP — Add sentences to cover missing phonemes
# Target: dˁ, tˁ, sˁ, zˁ, ʃ, ʒ, ħ, ʕ, j
# ============================================================

COVERAGE_SENTENCES = [

    # /ħ/ — voiceless pharyngeal (7 in Chat-Arabic)
    ("CS012", "7al x-as, ara k-sreɣ tura.",
     "Take it easy, I will help you now.", "confirmation"),

    # /ʃ/ — voiceless postalveolar (sh)
    ("CS013", "Shkel isem inek s tira.",
     "Please spell your name.", "triage"),

    # /dˁ/ + /tˁ/ — emphatic stops
    ("CS014", "Adhu n lmushkil yettwafhem.",
     "The nature of the problem is understood.", "triage"),

    # /sˁ/ — emphatic alveolar fricative
    ("CS015", "Ssuter aneɣmis s ufriwen.",
     "Request the message through the app.", "triage"),

    # /zˁ/ — voiced emphatic fricative
    ("CS016", "Aẓar n lmushkil d amiḍan.",
     "The root of the problem is the account.", "complaint"),

    # /ʒ/ — voiced postalveolar
    ("CS017", "Ljiha n umsawal tga taɣellist.",
     "The support department is available.", "confirmation"),

    # /ʕ/ — voiced pharyngeal (3 in Chat-Arabic)
    ("CS018", "3mmer umiḍan inek s lmeɛlumat.",
     "Fill your account with the information.", "triage"),

    # /j/ — palatal glide
    ("CS019", "Yiwen n wass ara k-d-aweɣ.",
     "I will contact you within one day.", "escalation"),
]

# Add to main corpus and recalculate
for row in COVERAGE_SENTENCES:
    id_, sentence, gloss_en, domain = row
    normalized = tarifit_text_normalizer(sentence)
    df_cs = df_cs._append({
        "id": id_, "domain": domain,
        "original": sentence, "normalized": normalized,
        "gloss_en": gloss_en
    }, ignore_index=True)

# Recalculate coverage
all_normalized = " ".join(df_cs["normalized"].tolist())
ipa_clean = re.sub(r"[\s\-]", "", all_normalized)

covered = [p for p in ALL_TASHELHIT if p in ipa_clean]
missing = [p for p in ALL_TASHELHIT if p not in ipa_clean]
coverage_pct = len(covered) / len(ALL_TASHELHIT) * 100

print("=" * 50)
print("UPDATED COVERAGE REPORT")
print("=" * 50)
print(f"Total sentences  : {len(df_cs)}")
print(f"Covered          : {len(covered)}")
print(f"Missing          : {len(missing)}")
print(f"Coverage         : {coverage_pct:.1f}%")

if missing:
    print("\nStill missing:")
    for p in missing:
        print(f"  /{p}/")
else:
    print("\n✅ All phonemes covered!")

# Export updated CSV
df_cs.to_csv("tarifit_customer_service_tts.csv", index=False, sep="|")
print("\n✅ CSV updated.")

In [ ]:
print("Still missing:")
for p in missing:
    for cat, phonemes in TASHELHIT_PHONEMES.items():
        if p in phonemes:
            print(f"  /{p}/  [{cat}]  {phonemes[p]}")

In [ ]:
# ============================================================
# STEP — Final sentences to reach 100% phoneme coverage
# Native speaker validated — Jamal Saghraoui
# ============================================================

FINAL_SENTENCES = [
    # /ʒ/ — voiced postalveolar
    ("CS020", "ljihaz war ikheddem",
     "The device is not working", "complaint"),

    # /tˁ/ — voiceless emphatic dental
    ("CS021", "adhbir yedhwa deg wjenna",
     "The pigeon flew in the sky", "other"),

    # /sˁ/ — voiceless emphatic alveolar
    ("CS022", "nhara aṣmmidh",
     "Today it is cold", "other"),

    # /zˁ/ — voiced emphatic alveolar
    ("CS023", "idammen d-iẓgwaghen",
     "The blood is red", "other"),
]

for row in FINAL_SENTENCES:
    id_, sentence, gloss_en, domain = row
    normalized = tarifit_text_normalizer(sentence)
    df_cs = df_cs._append({
        "id": id_, "domain": domain,
        "original": sentence, "normalized": normalized,
        "gloss_en": gloss_en
    }, ignore_index=True)

# Recalculate
all_normalized = " ".join(df_cs["normalized"].tolist())
ipa_clean = re.sub(r"[\s\-]", "", all_normalized)
covered = [p for p in ALL_TASHELHIT if p in ipa_clean]
missing = [p for p in ALL_TASHELHIT if p not in ipa_clean]
coverage_pct = len(covered) / len(ALL_TASHELHIT) * 100

print("=" * 50)
print("FINAL COVERAGE REPORT")
print("=" * 50)
print(f"Total sentences  : {len(df_cs)}")
print(f"Covered          : {len(covered)}")
print(f"Missing          : {len(missing)}")
print(f"Coverage         : {coverage_pct:.1f}%")

if missing:
    print("\nStill missing:")
    for p in missing:
        print(f"  /{p}/")
else:
    print("\n✅ All phonemes covered!")

df_cs.to_csv("tarifit_customer_service_tts.csv", index=False, sep="|")
print("✅ CSV updated.")

In [ ]:
# Diagnóstico — ver qué produce el normalizador
test_frases = [
    "ljihaz war ikheddem",
    "nhara aṣmmidh",
    "idammen d-iẓwaghen",
    "adhbir yedhwa deg wjenna",
]

print("DIAGNÓSTICO:")
for f in test_frases:
    resultado = tarifit_text_normalizer(f)
    print(f"  IN:  {f}")
    print(f"  OUT: {resultado}")
    print()

In [ ]:
# Final sentence — covers /j/ and /tˁ/
df_cs = df_cs._append({
    "id": "CS024",
    "domain": "other",
    "original": "y-uyar ghar thaddath",
    "normalized": tarifit_text_normalizer("y-uyar ghar thaddath"),
    "gloss_en": "he went home"
}, ignore_index=True)

# Recalculate
all_normalized = " ".join(df_cs["normalized"].tolist())
ipa_clean = re.sub(r"[\s\-]", "", all_normalized)
covered = [p for p in ALL_TASHELHIT if p in ipa_clean]
missing = [p for p in ALL_TASHELHIT if p not in ipa_clean]
coverage_pct = len(covered) / len(ALL_TASHELHIT) * 100

print("=" * 50)
print("FINAL COVERAGE REPORT")
print("=" * 50)
print(f"Total sentences  : {len(df_cs)}")
print(f"Covered          : {len(covered)}")
print(f"Missing          : {len(missing)}")
print(f"Coverage         : {coverage_pct:.1f}%")

if missing:
    print("\nStill missing:")
    for p in missing:
        print(f"  /{p}/")
else:
    print("\n✅ All phonemes covered!")

df_cs.to_csv("tarifit_customer_service_tts.csv", index=False, sep="|")
print("✅ CSV updated.")

In [ ]:
test = "y-uyar ghar thaddath"
resultado = tarifit_text_normalizer(test)
print(f"IN:  {test}")
print(f"OUT: {resultado}")
print()
print(f"¿Contiene 'j'? {'j' in resultado}")
print(f"¿Contiene 'ʒ'? {'ʒ' in resultado}")

In [ ]:
test = "y-uyar ghar thaddath"
resultado = tarifit_text_normalizer(test)
print(f"OUT: {resultado}")
print(f"¿Contiene 'y'? {'y' in resultado}")
print(f"¿Contiene 'j'? {'j' in resultado}")

In [75]:
# ============================================================
# STEP — Tifinagh to Latin Transliteration
# Converts standard IRCAM Tifinagh script to Latin
# for TTS normalization pipeline
# ============================================================

TIFINAGH_TO_LATIN = {
    # Vowels
    "ⴰ": "a",
    "ⵉ": "i",
    "ⵓ": "u",
    "ⴻ": "e",   # schwa / e

    # Stops
    "ⴱ": "b",
    "ⴷ": "d",
    "ⴹ": "ḍ",   # emphatic d
    "ⴳ": "g",
    "ⴽ": "k",
    "ⵇ": "q",
    "ⵜ": "t",
    "ⵟ": "ṭ",   # emphatic t

    # Fricatives
    "ⴼ": "f",
    "ⵙ": "s",
    "ⵚ": "ṣ",   # emphatic s
    "ⵣ": "z",
    "ⵥ": "ẓ",   # emphatic z
    "ⵛ": "sh",
    "ⵅ": "kh",
    "ⵖ": "gh",
    "ⵀ": "h",
    "ⵃ": "7",   # ħ — pharyngeal
    "ⵄ": "3",   # ʕ — pharyngeal voiced

    # Nasals
    "ⵎ": "m",
    "ⵏ": "n",

    # Liquids
    "ⵍ": "l",
    "ⵔ": "r",
    "ⵕ": "rr",  # emphatic r

    # Glides
    "ⵡ": "w",
    "ⵢ": "y",
}

def tifinagh_to_latin(text):
    """Convert Tifinagh script to Latin transliteration."""
    result = text
    for tifinagh, latin in TIFINAGH_TO_LATIN.items():
        result = result.replace(tifinagh, latin)
    return result

def tifinagh_to_tts(text):
    """Full pipeline: Tifinagh → Latin → Normalized phonetic."""
    latin = tifinagh_to_latin(text)
    normalized = tarifit_text_normalizer(latin)
    return latin, normalized

# ── Test ─────────────────────────────────────────────────
test_tifinagh = [
    "ⴰⵣⵓⵍ",           # Azul — hello
    "ⵜⴰⵎⴰⵣⵉⵖⵜ",       # Tamazight
    "ⴰⵔⵔⵉⴼ",           # Arrif — the Rif
    "ⵉⵎⴰⵣⵉⵖⴻⵏ",       # Imazighen
]

print("=" * 55)
print("TIFINAGH → LATIN → TTS PIPELINE")
print("=" * 55)
for word in test_tifinagh:
    latin, normalized = tifinagh_to_tts(word)
    print(f"  TIFINAGH : {word}")
    print(f"  LATIN    : {latin}")
    print(f"  TTS      : {normalized}")
    print()

TIFINAGH → LATIN → TTS PIPELINE
  TIFINAGH : ⴰⵣⵓⵍ
  LATIN    : azul
  TTS      : azul

  TIFINAGH : ⵜⴰⵎⴰⵣⵉⵖⵜ
  LATIN    : tamazight
  TTS      : tamaziɣt

  TIFINAGH : ⴰⵔⵔⵉⴼ
  LATIN    : arrif
  TTS      : arːif

  TIFINAGH : ⵉⵎⴰⵣⵉⵖⴻⵏ
  LATIN    : imazighen
  TTS      : imaziɣen

